# Phase 7 — 데이터 처리 및 통계 (예제)

측정 결과를 pandas 표로 정리해 통계·시각화·저장까지 해 봅니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; break
plt.rcParams['axes.unicode_minus'] = False

## 1. 측정 결과를 DataFrame 으로
Phase 6 에서 여러 위치를 측정했다고 가정하고, 그 결과를 표(DataFrame)에 담습니다. 리스트/딕셔너리로 흩어진 값을 표로 만들면 통계·필터·저장이 쉬워집니다.

In [ ]:
import pandas as pd, numpy as np
np.random.seed(11)

# 두 시료(A,B) 에서 위치별 두께(nm) 측정 결과 (이상치 2개 포함)
n=30
thick=np.round(np.random.normal(8.75,0.3,n),2)
thick[5]=10.9; thick[20]=6.7    # 이상치

df=pd.DataFrame({
    'pos': np.arange(1,n+1),
    'thick_nm': thick,
    'sample': ['A']*15 + ['B']*15,
})
df.head()

## 2. 열 선택 · 행 필터 · 정렬

In [ ]:
print(df['thick_nm'].head())          # 열 하나 선택
print()
print(df[df['thick_nm'] > 8.8].head())  # 조건으로 행 필터
print()
print(df.sort_values('thick_nm', ascending=False).head(3))  # 상위 3개

## 3. 기술통계
`describe()` 는 개수·평균·표준편차·분위수를 한 번에 보여줍니다.

In [ ]:
print('평균   :', round(df['thick_nm'].mean(), 3))
print('표준편차:', round(df['thick_nm'].std(), 3))
print('중앙값 :', df['thick_nm'].median())
print('1사분위:', df['thick_nm'].quantile(0.25))

df['thick_nm'].describe()

## 4. 그룹별 통계: groupby
시료(sample)별로 묶어 그룹마다 통계를 냅니다.

In [ ]:
print(df.groupby('sample')['thick_nm'].mean())
print()
df.groupby('sample')['thick_nm'].agg(['mean','std','count'])

## 5. 이상치 처리 (IQR)
튀는 값은 통계를 왜곡합니다. IQR(사분위 범위) 밖의 값을 걸러냅니다. 이상치 제거 전후 평균을 비교해 봅니다.

In [ ]:
q1=df['thick_nm'].quantile(0.25)
q3=df['thick_nm'].quantile(0.75)
iqr=q3-q1
lo,hi = q1-1.5*iqr, q3+1.5*iqr

clean=df[(df['thick_nm']>=lo) & (df['thick_nm']<=hi)]

print('제거 전 개수/평균 :', len(df), round(df['thick_nm'].mean(),3))
print('제거 후 개수/평균 :', len(clean), round(clean['thick_nm'].mean(),3))

## 6. 시각화
분포는 히스토그램·boxplot, 측정 순서에 따른 변화는 추세로 봅니다. (그래프 제목은 영어로 표기)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11,3))
ax[0].hist(df['thick_nm'], bins=12, color='#5EEAD4', edgecolor='#0D9488')
ax[0].set_title('histogram'); ax[0].set_xlabel('thickness (nm)')
ax[1].boxplot(df['thick_nm']); ax[1].set_title('boxplot'); ax[1].set_ylabel('thickness (nm)')
ax[2].plot(df['pos'], df['thick_nm'], '-o', ms=3); ax[2].axhline(df['thick_nm'].mean(), color='#0D9488')
ax[2].set_title('trend'); ax[2].set_xlabel('position')
plt.tight_layout(); plt.show()

## 7. 결과 내보내기
정리한 표를 CSV/Excel 로 저장합니다. (여기서는 임시 폴더에 저장)

In [ ]:
import tempfile, os
path=os.path.join(tempfile.gettempdir(), 'result.csv')
clean.to_csv(path, index=False)
print('저장 완료:', path)

# 엑셀로도 저장 가능 (openpyxl 필요)
# clean.to_excel('result.xlsx', index=False)
df.describe().to_csv(os.path.join(tempfile.gettempdir(),'summary.csv'))
print('요약 통계도 저장 완료')

## 8. 측정 → 표 → 요약 (전체 흐름)
Phase 6 의 반복 측정 결과 리스트를 그대로 표로 만들면, 위 과정을 한 흐름으로 자동화할 수 있습니다.

In [ ]:
# Phase 6 스타일의 결과 리스트 (위치, 두께nm)
results=[{'pos':x, 'thick_nm':round(np.random.normal(8.75,0.3),2)} for x in range(1,11)]
rep=pd.DataFrame(results)

print(rep['thick_nm'].agg(['mean','std','min','max']))
print('\n리포트 한 줄 요약:')
print(f"두께 {rep['thick_nm'].mean():.2f} ± {rep['thick_nm'].std():.2f} nm (n={len(rep)})")

---
예제 실행을 마친 후 `07_practice.ipynb` 의 문제를 해결하십시오.